In [0]:
%sql
CREATE OR REPLACE FUNCTION workspace.silver.normalize_city(s STRING)
RETURNS STRING
RETURN translate(lower(trim(s)), 'áàâãäéèêëíìîïóòôõöúùûüç', 'aaaaaeeeeiiiiooooouuuuc');

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.customers AS
SELECT customer_id,
       customer_unique_id,
       lpad(trim(customer_zip_code_prefix), 5, '0')      AS zip_prefix,
       workspace.silver.normalize_city(customer_city)    AS city,
       upper(trim(customer_state))                       AS state
FROM workspace.bronze.customers
WHERE customer_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.sellers AS
SELECT seller_id,
       lpad(trim(seller_zip_code_prefix), 5, '0')     AS zip_prefix,
       workspace.silver.normalize_city(seller_city)   AS city,
       upper(trim(seller_state))                      AS state
FROM workspace.bronze.sellers
WHERE seller_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY seller_id ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.order_items AS
SELECT order_id,
       try_cast(order_item_id AS INT)            AS order_item_id,
       product_id, seller_id,
       to_timestamp(shipping_limit_date)         AS shipping_limit_ts,
       try_cast(price AS DECIMAL(12,2))          AS price,
       try_cast(freight_value AS DECIMAL(12,2))  AS freight_value
FROM workspace.bronze.order_items
WHERE order_id IS NOT NULL
  AND try_cast(price AS DECIMAL(12,2)) >= 0
QUALIFY row_number() OVER (PARTITION BY order_id, order_item_id ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.order_payments AS
SELECT order_id,
       try_cast(payment_sequential AS INT)                  AS payment_sequential,
       lower(trim(payment_type))                            AS payment_type,
       greatest(try_cast(payment_installments AS INT), 1)   AS installments,
       try_cast(payment_value AS DECIMAL(12,2))             AS payment_value
FROM workspace.bronze.order_payments
WHERE order_id IS NOT NULL
  AND lower(trim(payment_type)) <> 'not_defined'
QUALIFY row_number() OVER (PARTITION BY order_id, payment_sequential ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.order_reviews AS
SELECT review_id, order_id,
       try_cast(review_score AS INT)                         AS review_score,
       nullif(trim(review_comment_title), '')                AS comment_title,
       nullif(trim(regexp_replace(review_comment_message, '[\\r\\n]+', ' ')), '') AS comment_message,
       to_timestamp(review_creation_date)                    AS created_ts,
       to_timestamp(review_answer_timestamp)                 AS answered_ts
FROM workspace.bronze.order_reviews
WHERE order_id IS NOT NULL
  AND try_cast(review_score AS INT) BETWEEN 1 AND 5
QUALIFY row_number() OVER (PARTITION BY order_id
                           ORDER BY to_timestamp(review_answer_timestamp) DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.products AS
SELECT p.product_id,
       coalesce(t.product_category_name_english, p.product_category_name, 'unknown') AS category,
       try_cast(p.product_name_lenght AS INT)         AS name_length,
       try_cast(p.product_description_lenght AS INT)  AS description_length,
       try_cast(p.product_photos_qty AS INT)          AS photos_qty,
       try_cast(p.product_weight_g AS INT)            AS weight_g,
       try_cast(p.product_length_cm AS INT)           AS length_cm,
       try_cast(p.product_height_cm AS INT)           AS height_cm,
       try_cast(p.product_width_cm AS INT)            AS width_cm
FROM workspace.bronze.products p
LEFT JOIN workspace.bronze.category_translation t
       ON trim(p.product_category_name) = trim(t.product_category_name)
WHERE p.product_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY p.product_id ORDER BY p._ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.geolocation AS
SELECT lpad(trim(geolocation_zip_code_prefix), 5, '0')                  AS zip_prefix,
       avg(try_cast(geolocation_lat AS DOUBLE))                         AS lat,
       avg(try_cast(geolocation_lng AS DOUBLE))                         AS lng,
       mode(workspace.silver.normalize_city(geolocation_city))          AS city,
       mode(upper(trim(geolocation_state)))                             AS state
FROM workspace.bronze.geolocation
WHERE try_cast(geolocation_lat AS DOUBLE) BETWEEN -34 AND 6
  AND try_cast(geolocation_lng AS DOUBLE) BETWEEN -74 AND -34
GROUP BY 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Filas por tabla, bronze vs silver
SELECT 'orders' t, (SELECT count(*) FROM workspace.bronze.orders) b, (SELECT count(*) FROM workspace.silver.orders) s
UNION ALL SELECT 'reviews', (SELECT count(*) FROM workspace.bronze.order_reviews), (SELECT count(*) FROM workspace.silver.order_reviews)
UNION ALL SELECT 'payments', (SELECT count(*) FROM workspace.bronze.order_payments), (SELECT count(*) FROM workspace.silver.order_payments);

-- Integridad: items sin orden, sin producto o sin seller (lo esperado es 0)
SELECT
  count_if(o.order_id IS NULL)  AS items_sin_orden,
  count_if(p.product_id IS NULL) AS items_sin_producto,
  count_if(s.seller_id IS NULL)  AS items_sin_seller
FROM workspace.silver.order_items i
LEFT JOIN workspace.silver.orders o   ON i.order_id = o.order_id
LEFT JOIN workspace.silver.products p ON i.product_id = p.product_id
LEFT JOIN workspace.silver.sellers s  ON i.seller_id = s.seller_id;

-- Cobertura geográfica: % de clientes con código postal encontrado
SELECT round(100 * avg(CASE WHEN g.zip_prefix IS NOT NULL THEN 1 ELSE 0 END), 2) AS pct_match
FROM workspace.silver.customers c
LEFT JOIN workspace.silver.geolocation g ON c.zip_prefix = g.zip_prefix;

pct_match
99.72
